# 독버섯 분류 프로젝트 - RAG 파트
CV 모델이 예측한 class_code를 받아 독성 정보·닮은 식용버섯 구분법·자유 질의응답을 제공하는 RAG 서비스 구축 노트북입니다.

| 단계 | 내용 |
|---|---|
| 1 | 설치 및 임포트 |
| 2 | Drive 마운트 및 API 키 설정 |
| 3 | 생태도감 PDF 페이지 매핑 |
| 4 | 종별 정보 파싱 (mushroom_db 구축) |
| 5 | FAISS 벡터스토어 구축 (자유 질문용) |
| 6 | RAG 함수 정의 |
| 7 | 전체 동작 검증 |
| 8 | 백엔드 전달용 파일 정리 |

## 1. 설치 및 라이브러리 임포트

In [15]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu pdfplumber -q

In [16]:
import os, re, json
import pdfplumber
from google.colab import drive, userdata
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

## 2. Drive 마운트 및 API 키 설정
- Colab 왼쪽 열쇠 아이콘에 `GOOGLE_API_KEY`가 등록되어 있어야 합니다.
- `PDF_PATH`는 국립수목원 「우리나라 독버섯」 생태도감 PDF 경로입니다.

In [17]:
drive.mount('/content/drive')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

DATA_DIR = '/content/drive/MyDrive/PoisonMushroom/'
PDF_PATH = f'{DATA_DIR}/우리나라 독버섯 생태도감 [개정판].pdf'

TARGET_TOXIC = ['마귀광대버섯', '개나리광대버섯', '삿갓외대버섯', '화경솔밭버섯',
                 '노란개암버섯', '독우산광대버섯', '붉은사슴뿔버섯']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


## 3. 생태도감 PDF 페이지 매핑
- 리스트(인쇄 페이지 68~73)에는 도감 수록 194종 전체의 `국명 ... p.쪽수`가 나열되어 있습니다. 이를 파싱해 `species_page_index_full`(국명 → 설명 시작페이지)을 만듭니다.
- 설명 본문(인쇄 페이지 74~461)은 **한 종이 2페이지**를 차지하는 구조입니다. (506페이지 이후 구간은 사용하지 않습니다.)
- CV로 분류하는 7종은 미리 파싱해 `mushroom_db`에 저장(빠른 응답), 나머지 종은 챗봇에서 질문이 들어올 때 그때 페이지를 찾아 파싱합니다(지연 검색).

In [19]:
#pdf 인덱스 -> 인쇄 페이지
def pdf_to_printed_pages(pdf_index: int) -> tuple[int, int]:
    """PDF 파일 인덱스(0-based) -> 해당 페이지에 인쇄된 책 페이지 번호 2개(좌/우)"""
    pdf_number = pdf_index + 1  # 1-based PDF 페이지 번호
    return pdf_number * 2 - 2, pdf_number * 2 - 1


#인쇄 페이지 -> pdf 인덱스
def printed_to_pdf_index(printed_page: int) -> int:
    """인쇄 페이지 번호 -> PDF 파일 인덱스(0-based)"""
    return (printed_page + 2) // 2 - 1

In [20]:
# 독버섯 리스트(인쇄 페이지 68~73)에서 194종 전체 {국명: 설명시작페이지} 매핑 생성
def build_species_page_index_from_list(pdf_path, start_printed_page=68, end_printed_page=73):
    """'No. 국명 문명 강명 목명 과명 학명 p.쪽수' 리스트 구간을 파싱"""
    start_idx = printed_to_pdf_index(start_printed_page)
    end_idx = printed_to_pdf_index(end_printed_page) + 1

    text_all = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i in range(start_idx, min(end_idx, len(pdf.pages))):
            text_all += (pdf.pages[i].extract_text() or "") + "\n"

    # 예: "12 검은띠말똥버섯 ... p.556" -> ("검은띠말똥버섯", "556")
    pattern = r'\d+\s+([가-힣]+버섯)\s+.*?p\.(\d+)'
    matches = re.findall(pattern, text_all, re.DOTALL)
    return {name: int(page) for name, page in matches}


species_page_index_full = build_species_page_index_from_list(PDF_PATH, 68, 73)
print(f"리스트에서 추출된 전체 종 수: {len(species_page_index_full)}")  # 194에 근접해야 함

# 461페이지 이후 구간(506~ 등)은 사용하지 않으므로 제외
species_page_index_full = {name: p for name, p in species_page_index_full.items() if p <= 460}
print(f"74~461 구간(사용 대상) 종 수: {len(species_page_index_full)}")
print({k: species_page_index_full[k] for k in TARGET_TOXIC if k in species_page_index_full})


리스트에서 추출된 전체 종 수: 208
74~461 구간(사용 대상) 종 수: 178
{'마귀광대버섯': 160, '개나리광대버섯': 180, '삿갓외대버섯': 212, '화경솔밭버섯': 276, '노란개암버섯': 296, '독우산광대버섯': 190, '붉은사슴뿔버섯': 86}


## 4. 종별 정보 파싱 (74~461페이지, 종당 2페이지 구조)
CV로 분류하는 7종은 여기서 미리 파싱해 `mushroom_db`에 저장합니다(예측 직후 빠른 응답용).
나머지 187종은 여기서 파싱하지 않고, 챗봇이 해당 종명을 언급할 때 `search_species_by_name`으로 그때 찾습니다.

In [21]:
def get_species_info_v2(pdf, printed_page: int, target_name: str) -> dict:
    """74~461페이지 전용: 한 종이 printed_page, printed_page+1 두 페이지에 걸쳐 있음"""
    idx1 = printed_to_pdf_index(printed_page)
    idx2 = printed_to_pdf_index(printed_page + 1)

    text = (pdf.pages[idx1].extract_text() or "")
    if idx2 != idx1 and idx2 < len(pdf.pages):
        text += "\n" + (pdf.pages[idx2].extract_text() or "")

    # "7 붉은사슴뿔버섯" 처럼 번호+종명이 나오는 위치를 찾아 그 지점부터만 사용
    # (헤더/분류명 길이가 종마다 달라서 고정폭 text[:50] 방식은 신뢰할 수 없음)
    m = re.search(r'\d+\s*' + re.escape(target_name), text)
    if not m:
        return None
    text = text[m.start():]

    def extract(pattern):
        r = re.search(pattern, text, re.DOTALL)
        return r.group(1).strip() if r else None

    return {
        "name_kr": target_name,
        "name_sci": extract(r'([A-Z][a-z]+ [a-z]+ \([^)]+\))'),
        "phylum": extract(r'문명\s*Phylum\s*(.*?)\s*강명'),
        "class_": extract(r'강명\s*Class\s*(.*?)\s*목명'),
        "order": extract(r'목명\s*Order\s*(.*?)\s*과명'),
        "family": extract(r'과명\s*Family\s*(.*?)\s*속명'),
        "genus": extract(r'속명\s*Genus\s*(.*?)\s*외부 형태적 특징'),
        "morphology": extract(r'외부 형태적 특징\s*(.*?)\s*현미경적 특징'),
        "microscopic": extract(r'현미경적 특징\s*(.*?)\s*발생시기'),
        "onset_time": extract(r'발생시기\s*(.*?)\s*발생장소'),
        "habitat": extract(r'발생장소\s*(.*?)\s*분포'),
        "distribution": extract(r'분포\s*(.*?)\s*비슷한 식용버섯류'),
        "lookalike_edible": extract(r'비슷한 식용버섯류\s*(.*?)\s*p\.\d+'),
        # 중독유형/주요독소/중독증상/독성강도는 헤더 4개가 붙어 나온 뒤 값이 나오는 구조라
        # 개별 필드로 안전하게 못 자르므로, 원문 블록 전체를 저장해두고 치료방법 섹션과 매칭시켜 채운다.
        "raw_toxicity_block": extract(r'중독유형\s*주요독소\s*중독증상\s*독성강도\s*(.*)$'),
    }


In [22]:
# 7종 타겟만 미리 파싱 (빠른 응답용)
species_page_index = {name: species_page_index_full[name] for name in TARGET_TOXIC if name in species_page_index_full}

mushroom_db = {}
with pdfplumber.open(PDF_PATH) as pdf:
    for name, page in species_page_index.items():
        info = get_species_info_v2(pdf, page, name)
        if info:
            mushroom_db[name] = info

print(f"사전 파싱 완료: {len(mushroom_db)} / {len(TARGET_TOXIC)}")


def search_species_by_name(name: str) -> dict:
    """7종 외 나머지 종을 챗봇이 질문받은 시점에 실시간으로 찾아 파싱"""
    page = species_page_index_full.get(name)
    if not page:
        return {"error": f"'{name}' 정보를 리스트에서 찾을 수 없습니다."}
    with pdfplumber.open(PDF_PATH) as pdf:
        info = get_species_info_v2(pdf, page, name)
    return info if info else {"error": f"'{name}' 페이지 파싱 실패 (p.{page})"}


사전 파싱 완료: 7 / 7


### 4-1. 파싱 검증
타겟 7종 모두 필수 필드가 채워졌는지 확인합니다.

In [23]:
for name in TARGET_TOXIC:
    info = mushroom_db.get(name)
    assert info is not None, f"파싱 실패: {name}"
    assert info.get('morphology'), f"형태 정보 없음: {name}"
    print(f"OK: {name} -> 비슷한 식용버섯: {info.get('lookalike_edible')}")


OK: 마귀광대버섯 -> 비슷한 식용버섯: 메스꺼움, 구토,
맛광대버섯 (Amanita esculenta).
OK: 개나리광대버섯 -> 비슷한 식용버섯: 설사, 복통, 심한
노란달걀버섯 (Amanita javanica).
OK: 삿갓외대버섯 -> 비슷한 식용버섯: 외대덧버섯 (Entoloma sarcopum).
OK: 화경솔밭버섯 -> 비슷한 식용버섯: 산느타리 (Pleurotus pulmonarius).
OK: 노란개암버섯 -> 비슷한 식용버섯: 고혈압, 반사항진,
정신불안, 인지장
개암버섯 (Hypholoma lateritium).
OK: 독우산광대버섯 -> 비슷한 식용버섯: 용혈, 구토, 설사,
마비 등
흰주름버섯 (Agaricus arvensis).
OK: 붉은사슴뿔버섯 -> 비슷한 식용버섯: 중독증상
불로초(영지) (Ganoderma lucidum).


In [24]:
# 목차(PDF 인덱스 6) 파싱이 실제 PDF의 dot-leader 문자와 안 맞아 자동 추출이 실패하여, 확인된 목차 내용을 직접 입력합니다.
toc_index = {
    "책을 펴내며": 4,
    "일러두기": 8,
    "일러두기 내용": 8,
    "독버섯 중독유형별 모식도": 10,
    "독버섯 랭크 (치명적>위험>유해>약독)": 12,
    "Ⅰ. 독버섯중독사고 예방 및 치료": 26,
    "1. 독버섯 중독사고 예방방법": 26,
    "2. 독버섯의 주요 중독유형 및 치료방법": 27,
    "Ⅱ. 독버섯 이야기": 34,
    "Ⅲ. 국내기록 독버섯 (234종)": 63,
    "1. 국내기록독버섯(194종)": 74,
    "2. 국내기록종 미확보 또는 존재가 불확실한 독버섯(40종)": 462,
    "Ⅳ. 독버섯중독사고와 관련된 주요 식용버섯 (72종)": 503,
    "Ⅴ. 참고문헌": 650,
    "Ⅵ. 용어설명": 651,
    "Ⅶ. 찾아보기": 656,
}
print(f"목차 항목 수: {len(toc_index)}")

# '치료방법' 섹션 범위: 27 ~ 33페이지 (다음 항목 'Ⅱ. 독버섯 이야기'가 34이므로)
TREATMENT_START_PAGE = 27
TREATMENT_END_PAGE = 33


def find_treatment_pages(pdf_path, start_printed_page, end_printed_page):
    """치료방법 섹션(인쇄 페이지 기준)의 실제 PDF 인덱스를 찾아 텍스트로 반환"""
    start_idx = printed_to_pdf_index(start_printed_page)
    end_idx = printed_to_pdf_index(end_printed_page) + 1

    with pdfplumber.open(pdf_path) as pdf:
        idxs = list(range(start_idx, min(end_idx, len(pdf.pages))))
        text = "\n".join((pdf.pages[i].extract_text() or "") for i in idxs)
    return idxs, text


treatment_idx, treatment_text = find_treatment_pages(PDF_PATH, TREATMENT_START_PAGE, TREATMENT_END_PAGE)
print("치료방법 PDF 인덱스:", treatment_idx)
print(treatment_text[:500])


목차 항목 수: 16
치료방법 PDF 인덱스: [13, 14, 15, 16]
2. 독버섯의 주요 중독유형 및 치료방법
Ⅰ 독버섯 중독사고 예방 및 치료
독버섯을 얼마나 먹어야 중독 증상을 나타내는 지에 대해서는 거의 연구가 없습니다.
또한 버섯의 크기, 생장단계, 다른 균이나 벌레 등의 오염, 독성성분, 개인차 등의 변수가
많아 정량화하는 것이 어렵고, 독버섯을 쉽게 채집하여 실험해 볼 수 있는 여건 역시 조성
되어 있지 않습니다. 다만 일부 독버섯 유래 독성분을 쥐를 이용한 반수치사량(LD50: 실험
1. 독버섯 중독사고 예방방법 동물 개체수의 50%을 죽음에 이르게 하는데 필요한 물질량) 실험결과가 있습니다. 동물
의 체중 1㎏에 대해 독성물질의 양(㎎)으로 나타냅니다. 다음에 제시된 독성물질의 반사
야생에서 독버섯을 채집하여 먹는 경우, 확실히 동정이 된 버섯을 제외하고는 먹지
치사량을 바탕으로 독성분의 강도를 대략적으로 추측만 가능할 뿐이므로 참고자료로만
않는 것이 가장 좋습니다. 또한 식용으로 알려져 있는 버섯이라고 해도 생식하는 것보다
활용해야 할


In [25]:
# 치료방법 원문을 LLM에게 넘겨 {중독유형: {rank, emergency_action}} 형태로 구조화
treatment_parse_prompt = f"""아래는 독버섯 도감의 '중독유형별 치료방법' 섹션 원문입니다.

{treatment_text}

이 내용을 바탕으로 중독유형별 정보를 JSON으로만 응답하세요 (설명, 코드블록 없이 순수 JSON):
{{
  "중독유형명1": {{"rank": "치명적/위험/유해/약독 중 하나", "emergency_action": "응급처치 요약 1문장"}},
  "중독유형명2": {{"rank": "...", "emergency_action": "..."}}
}}"""

response = llm.invoke(treatment_parse_prompt).content
print("LLM 원본 응답:\n", response)

try:
    treatment_info = json.loads(response.replace("```json", "").replace("```", "").strip())
except json.JSONDecodeError as e:
    print("JSON 파싱 실패:", e)
    treatment_info = {}

print(treatment_info)


LLM 원본 응답:
 ```json
{
  "아마톡신(amatoxin) 중독": {
    "rank": "치명적",
    "emergency_action": "장내 잔존물 제거를 위해 구토유발, 위세척, 설사제를 사용하며, 활성탄 투여와 수액 및 전해질 보충이 필요하고, 급성간부전 발생 시 전문 치료 및 간이식을 고려해야 합니다."
  },
  "무스카린(muscarin) 중독": {
    "rank": "위험",
    "emergency_action": "구토유발, 위세척, 활성탄, 설사제 투여는 도움이 되지 않으며, 기관지 분비물이나 심한 서맥 조절을 위해 아트로핀을 신중하게 투여하고, 증상 관리가 중요합니다."
  },
  "이보텐산-무시몰(ibotenic acid-muscimol) 중독": {
    "rank": "치명적",
    "emergency_action": "위세척과 활성탄 투여는 권장되지 않으며, 필요시 기도관리 및 인공환기를 시행하고 적절한 진정 상태를 유지하며 자연 회복되도록 증상을 관리해야 합니다."
  },
  "지로미트린(gyromitrin) 중독": {
    "rank": "위험",
    "emergency_action": "활성탄 투여는 도움이 되지 않으며, 발작 시 벤조디아제핀 또는 피리도신을 투여하고, 급성간부전 발생 시 전문 치료 및 간이식이 필요합니다."
  },
  "코프린(coprine) 중독": {
    "rank": "위험",
    "emergency_action": "활성탄 투여는 도움이 되지 않으며, 저혈압 발생 시 수액 정주 및 혈압상승제 투여, 심실상빈맥 시 약물 투여가 필요하며, 증상 호전까지 상태를 주시해야 합니다."
  },
  "환각(psilocybin-psilocin) 중독": {
    "rank": "유해",
    "emergency_action": "위세척과 활성탄 투여는 권장되지 않으며, 환자를 조용한 환경에 두어 진정시키고, 발작 시 벤조디아제핀을 투여하며, 소아 고열 시

In [26]:
def apply_treatment_info(mushroom_db, treatment_info):
    """각 종의 raw_toxicity_block을 치료방법 카테고리와 매칭해 rank/emergency_action 채우기"""
    for name, info in mushroom_db.items():
        raw = info.get("raw_toxicity_block", "") or ""
        matched = None
        for category in treatment_info:
            if category[:2] and category[:2] in raw:  # 카테고리명 앞 2글자로 느슨하게 매칭
                matched = category
                break
        if matched:
            info["rank"] = treatment_info[matched].get("rank")
            info["emergency_action"] = treatment_info[matched].get("emergency_action")
        else:
            info["rank"] = None
            info["emergency_action"] = "즉시 의료기관 방문"  # 매칭 실패 시 기본값
    return mushroom_db


mushroom_db = apply_treatment_info(mushroom_db, treatment_info)
for name, info in mushroom_db.items():
    print(name, "→ rank:", info.get("rank"), "| action:", info.get("emergency_action"))


마귀광대버섯 → rank: None | action: 즉시 의료기관 방문
개나리광대버섯 → rank: None | action: 즉시 의료기관 방문
삿갓외대버섯 → rank: None | action: 즉시 의료기관 방문
화경솔밭버섯 → rank: None | action: 즉시 의료기관 방문
노란개암버섯 → rank: None | action: 즉시 의료기관 방문
독우산광대버섯 → rank: None | action: 즉시 의료기관 방문
붉은사슴뿔버섯 → rank: None | action: 즉시 의료기관 방문


## 5. FAISS 벡터스토어 구축 (자유 질문용)
정형 필드가 아닌 자유 질문(예: 응급 대처 상세, 증상 진행 등)에 답하기 위한 벡터 인덱스를 구축합니다.

In [27]:
documents = []
for name, info in mushroom_db.items():
    content = f"""종명: {name} ({info.get('name_sci', '')})
외부 형태적 특징: {info.get('morphology', '')}
발생시기: {info.get('onset_time', '')}
발생장소: {info.get('habitat', '')}
비슷한 식용버섯: {info.get('lookalike_edible', '')}
독성 랭크: {info.get('rank', '')}
독성 유형: {info.get('toxin_type', '')}
중독증상: {info.get('symptoms', '')}
응급 대처: {info.get('emergency_action', '')}"""
    documents.append(Document(page_content=content, metadata={"name_kr": name}))

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local(f'{DATA_DIR}/faiss_index')

print("FAISS 벡터스토어 구축 및 저장 완료 (7종 기준)")


FAISS 벡터스토어 구축 및 저장 완료 (7종 기준)


## 6. RAG 함수 정의
| 함수 | 방식 | 용도 |
|---|---|---|
| `get_core_info` | dict 조회 | 예측 직후 즉시 표시되는 핵심 안전 정보 |
| `get_lookalike_comparison` | dict 매칭 + LLM 요약 | 닮은 식용버섯 구분법 |
| `ask_followup` | FAISS 검색 + LLM | 자유 질문 챗봇 |

In [28]:
def get_core_info(class_code: str) -> dict:
    """CV 예측 직후 즉시 표시되는 핵심 정보 (7종 전용, mushroom_db 조회)"""
    info = mushroom_db.get(class_code)
    if info is None:
        return {"error": "정보를 찾을 수 없습니다"}
    return {
        "name_kr": info['name_kr'],
        "name_sci": info.get('name_sci'),
        "rank": info.get('rank'),
        "toxin_type": info.get('toxin_type'),
        "onset_time": info.get('onset_time'),
        "emergency_action": info.get('emergency_action', "즉시 의료기관 방문"),
    }


def get_lookalike_comparison(class_code: str) -> str:
    """비슷한 식용버섯과의 구분법 (7종 전용)"""
    info = mushroom_db.get(class_code)
    if not info or not info.get('lookalike_edible'):
        return "비슷한 식용버섯 정보가 없습니다."

    prompt = f"""독버섯 '{info['name_kr']}' 특징:
{info.get('morphology', '')}

비슷한 식용버섯 정보(도감 원문): {info['lookalike_edible']}

두 종을 구분할 수 있는 핵심 포인트 2~3가지를 한국어로 간결하게 알려줘."""
    return llm.invoke(prompt).content


def ask_followup(name_kr: str, question: str) -> str:
    """자유 질문 챗봇. 7종은 사전 구축된 FAISS에서, 나머지 187종은 그 자리에서 검색해 답변"""
    if name_kr in mushroom_db:
        docs = vectorstore.similarity_search(f"{name_kr} {question}", k=3)
        context = "\n\n".join([d.page_content for d in docs])
    else:
        info = search_species_by_name(name_kr)
        if "error" in info:
            return info["error"]
        context = f"""종명: {name_kr} ({info.get('name_sci', '')})
외부 형태적 특징: {info.get('morphology', '')}
발생시기: {info.get('onset_time', '')}
발생장소: {info.get('habitat', '')}
비슷한 식용버섯: {info.get('lookalike_edible', '')}
독성 유형: {info.get('toxin_type', '')}
중독증상: {info.get('symptoms', '')}"""

    prompt = f"""아래는 독버섯 생태도감 발췌입니다.
{context}

사용자가 '{name_kr}'에 대해 다음과 같이 질문했습니다: "{question}"
생태도감 내용을 바탕으로 답변하고, 확실하지 않으면 반드시 의료기관 문의를 안내해줘."""
    return llm.invoke(prompt).content


## 7. 전체 동작 검증
타겟 7종에 대해 3개 함수가 모두 정상 동작하는지 확인합니다.

In [29]:
for name in TARGET_TOXIC:
    print(f"=== {name} ===")
    print(json.dumps(get_core_info(name), ensure_ascii=False, indent=2))
    print("- 비슷한 식용버섯 구분법:", get_lookalike_comparison(name)[:80], "...")
    print()

# 7종 외 종에 대한 지연 검색 예시
example_other = next(iter(set(species_page_index_full) - set(TARGET_TOXIC)), None)
if example_other:
    print(f"=== (7종 외) {example_other} 지연 검색 테스트 ===")
    print(json.dumps(search_species_by_name(example_other), ensure_ascii=False, indent=2))


=== 마귀광대버섯 ===
{
  "name_kr": "마귀광대버섯",
  "name_sci": "Amanita pantherina (DC.)",
  "rank": null,
  "toxin_type": null,
  "onset_time": "stizolobinic acid,\n여름~가을 amatoxine,\nallglycine,\npropargylglycine",
  "emergency_action": "즉시 의료기관 방문"
}
- 비슷한 식용버섯 구분법: 제공된 정보에는 독버섯 '마귀광대버섯'의 특징만 상세히 기술되어 있으며, '맛광대버섯'에 대한 구체적인 설명은 없습니다. 따라서 주어진 텍스트만 ...

=== 개나리광대버섯 ===
{
  "name_kr": "개나리광대버섯",
  "name_sci": null,
  "rank": null,
  "toxin_type": null,
  "onset_time": "amatoxin 등\n여름~가을",
  "emergency_action": "즉시 의료기관 방문"
}
- 비슷한 식용버섯 구분법: 제공된 정보에 따르면, '노란달걀버섯'에 대한 구체적인 형태적 특징은 없으며, 단지 '식용'이라는 점만 명시되어 있습니다. 따라서 '개나리광대버 ...

=== 삿갓외대버섯 ===
{
  "name_kr": "삿갓외대버섯",
  "name_sci": "Entoloma rhodopolium (Fr.)",
  "rank": null,
  "toxin_type": null,
  "onset_time": "질, choline,\n여름~가을\nmuscarin,\nvinylglycine 등",
  "emergency_action": "즉시 의료기관 방문"
}
- 비슷한 식용버섯 구분법: 제공해주신 정보에는 독버섯 '삿갓외대버섯'의 특징만 상세히 기술되어 있고, 식용버섯 '외대덧버섯'에 대한 구체적인 특징은 이름만 언급되어 있습니 ...

=== 화경솔밭버섯 ===
{
  "name_kr": "화경솔밭버섯",
  "name_sci"

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 27.402570327s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [31]:
# 자유 질문 예시 테스트
example_answer = ask_followup("독우산광대버섯", "먹은 지 3시간 지났는데 괜찮나요?")
print(example_answer)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 10.403772027s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## 8. 백엔드 전달용 파일 정리
아래 파일/폴더를 백엔드 팀장에게 전달하면 `/predict`, `/lookalike`, `/ask` API에 바로 연결할 수 있습니다.

| 파일 | 용도 |
|---|---|
| `mushroom_db.json` | `get_core_info`, `get_lookalike_comparison`에서 사용하는 고정 데이터 |
| `faiss_index/` | `ask_followup`에서 사용하는 벡터 인덱스 |
| 이 노트북의 6번 셀 함수 3개 | 백엔드 `main.py`에 그대로 이식 |

In [ ]:
import os
print("mushroom_db.json 존재:", os.path.exists(f'{DATA_DIR}/mushroom_db.json'))
print("faiss_index 폴더 존재:", os.path.exists(f'{DATA_DIR}/faiss_index'))